# ESM-2 for Enzyme Function Prediction

**Fully self-contained** — upload this single `.ipynb` to Google Colab and run all cells in order.

Pipeline:
1. Install dependencies
2. Download enzyme data from UniProt
3. Explore the dataset
4. Extract ESM-2 embeddings
5. Train classifiers (sklearn baselines + MLP)
6. Generate figures

Total runtime on Colab GPU: ~60-90 minutes.

In [ ]:
# ============================================================
# CELL 1: Install dependencies (run once)
# ============================================================
!pip install fair-esm -q
!pip install pandas numpy scikit-learn matplotlib seaborn requests tqdm -q

import os
import sys
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Download Data

In [ ]:
# ============================================================
# CELL 2: Download enzyme sequences from UniProt
#
# EC Level 1 classes:
#   1 - Oxidoreductases   4 - Lyases
#   2 - Transferases      5 - Isomerases
#   3 - Hydrolases        6 - Ligases
#
# Query: reviewed (Swiss-Prot) entries with EC number,
#        sequence length <= 1022 (ESM-2 context limit)
# ============================================================
import pandas as pd
import requests
import time

MAX_SEQ_LENGTH = 1022
EC_CLASS_NAMES = {
    1: "Oxidoreductases", 2: "Transferases", 3: "Hydrolases",
    4: "Lyases", 5: "Isomerases", 6: "Ligases"
}

def query_uniprot(ec_class, max_results=1500):
    """Query UniProt REST API for reviewed enzymes of one EC class."""
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    query = f"(reviewed:true) AND (ec:{ec_class}.*.*) AND (length:[* TO {MAX_SEQ_LENGTH}])"
    params = {
        "query": query,
        "format": "json",
        "fields": "accession,sequence,ec,protein_name,organism_name,length",
        "size": min(max_results, 500),
    }
    proteins = []
    next_link = None
    while True:
        url = next_link or base_url
        resp = requests.get(url, params=params if not next_link else None, timeout=30)
        if resp.status_code != 200:
            print(f"  HTTP {resp.status_code} for EC class {ec_class}")
            break
        data = resp.json()
        for entry in data.get("results", []):
            seq = entry.get("sequence", {}).get("value", "")
            if seq and len(seq) <= MAX_SEQ_LENGTH and len(seq) >= 20:
                ec_list = entry.get("ec", [])
                proteins.append({
                    "accession": entry.get("primaryAccession", ""),
                    "sequence": seq,
                    "ec_class": ec_class,
                    "ec_number": ec_list[0] if ec_list else f"{ec_class}.-.-.-",
                    "ec_level1_name": EC_CLASS_NAMES[ec_class],
                    "organism": entry.get("organism", {}).get("scientificName", "Unknown"),
                    "length": len(seq),
                })
        link = resp.headers.get("Link", "")
        if 'rel="next"' in link:
            next_link = link.split(";")[0].strip("<>")
            params = None
            time.sleep(0.5)
        else:
            break
        if len(proteins) >= max_results:
            break
    return proteins

def download_all_enzymes(target_per_class=1500):
    """Download enzymes for all 6 EC classes, return a DataFrame."""
    all_proteins = []
    for ec_class in range(1, 7):
        print(f"Downloading EC{ec_class} ({EC_CLASS_NAMES[ec_class]})...")
        proteins = query_uniprot(ec_class, max_results=target_per_class)
        all_proteins.extend(proteins)
        print(f"  Got {len(proteins)} sequences")
        time.sleep(1)
    df = pd.DataFrame(all_proteins)
    df.to_csv("enzymes_swissprot.csv", index=False)
    print(f"\nTotal: {len(df)} sequences saved to enzymes_swissprot.csv")
    print(df["ec_class"].value_counts().sort_index())
    return df

df = download_all_enzymes(target_per_class=1500)
df.head()

## Step 2: Explore the Dataset

In [ ]:
# ============================================================
# CELL 3: Explore dataset
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

EC_NAMES = {
    1: "Oxidoreductases", 2: "Transferases", 3: "Hydrolases",
    4: "Lyases", 5: "Isomerases", 6: "Ligases"
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Class distribution
counts = df["ec_class"].value_counts().sort_index()
axes[0].bar([f"EC{i}" for i in counts.index], counts.values,
            color=plt.cm.Set2(np.linspace(0, 1, 6)))
axes[0].set_title("EC Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha="center", fontweight="bold")

# Length distribution
axes[1].hist(df["length"], bins=50, color="#2196F3", edgecolor="white")
axes[1].axvline(df["length"].median(), color="red", linestyle="--",
                label=f'Median: {df["length"].median():.0f}')
axes[1].set_title("Sequence Length Distribution")
axes[1].set_xlabel("Length (aa)")
axes[1].legend()

plt.tight_layout()
os.makedirs("results/figures", exist_ok=True)
plt.savefig("results/figures/dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 3: Extract ESM-2 Embeddings

In [ ]:
# ============================================================
# CELL 4: Extract protein embeddings with ESM-2
#
# What ESM-2 does:
#   - Transformer trained on 250M protein sequences (masked LM)
#   - Learns evolutionary/functional patterns from sequence alone
#   - Outputs per-residue embeddings (480-dim for the 35M model)
#
# We use mean pooling: average all amino-acid embeddings to get
# one fixed-size vector per protein.
# ============================================================
import esm
import numpy as np
from tqdm import tqdm

MODEL_NAME = "esm2_t12_35M_UR50D"  # 35M params, 12 layers, 480-dim
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading ESM-2...")
model, alphabet = esm.pretrained.load_model_and_alphabet(MODEL_NAME)
model = model.to(device)
model.eval()
batch_converter = alphabet.get_batch_converter()
if hasattr(model, "args"):
    num_layers = model.args.num_layers
    embed_dim = model.args.embed_dim
else:
    num_layers = model.num_layers
    embed_dim = model.embed_dim
print(f"  Model loaded: {embed_dim}-dim embeddings, {num_layers} layers")
print(f"  Device: {device}")

# ---- QUICK TEST KNOB ----
# Set N_SAMPLES to a small number (e.g. 200) first to check speed.
# If each batch takes <2s, set back to the full count (9000).
N_SAMPLES = len(df)
seqs = df["sequence"].tolist()[:N_SAMPLES]
lbl = (df["ec_class"].values - 1).astype(np.int64)[:N_SAMPLES]

def extract_embeddings(sequences, batch_size=32):
    """Return mean-pooled ESM-2 embeddings, shape (n, 480)."""
    all_embeddings = []
    for i in tqdm(range(0, len(sequences), batch_size), desc="Embeddings"):
        batch_seqs = sequences[i:i + batch_size]
        data = [(f"seq_{j}", s) for j, s in enumerate(batch_seqs)]
        _, _, tokens = batch_converter(data)
        tokens = tokens.to(device)
        with torch.no_grad():
            res = model(tokens, repr_layers=[num_layers], return_contacts=False)
        reps = res["representations"][num_layers]
        for j in range(len(batch_seqs)):
            seq_len = len(batch_seqs[j])
            emb = reps[j, 1:seq_len + 1].mean(dim=0)  # mean pool, skip BOS/EOS
            all_embeddings.append(emb.cpu().numpy())
    return np.array(all_embeddings, dtype=np.float32)

print(f"Extracting embeddings for {len(seqs)} sequences...")
embeddings = extract_embeddings(seqs)
np.save("esm2_embeddings.npy", embeddings)
labels = lbl  # 0-indexed
np.save("labels.npy", labels)
print(f"Embeddings: {embeddings.shape}  Labels: {labels.shape}")

## Step 4: Train Classifiers

In [ ]:
# ============================================================
# CELL 5: Dataset split
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score

embeddings = np.load("esm2_embeddings.npy")
labels = np.load("labels.npy")

X_trainval, X_test, y_trainval, y_test = train_test_split(
    embeddings, labels, test_size=0.15, stratify=labels, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.176, stratify=y_trainval, random_state=42
)

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")
EC_CLASSES = [f"EC{i}" for i in range(1, 7)]

In [ ]:
# ============================================================
# CELL 6: Sklearn baselines (Logistic Regression, Random Forest, GB)
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

baselines = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=20,
                                            min_samples_leaf=5, class_weight="balanced",
                                            random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=300, max_depth=5,
                                                    learning_rate=0.1, subsample=0.8,
                                                    random_state=42),
}

sklearn_results = {}
for name, model in baselines.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    sklearn_results[name] = {
        "accuracy": acc, "f1_macro": f1,
        "report": classification_report(y_test, preds, output_dict=True),
    }
    print(f"  Accuracy: {acc:.4f}  Macro F1: {f1:.4f}")
    print(classification_report(y_test, preds, target_names=EC_CLASSES))

In [ ]:
# ============================================================
# CELL 7: MLP classifier (PyTorch) on frozen ESM-2 embeddings
#
# Architecture:
#   480 → Linear(256) → BN → ReLU → Dropout
#       → Linear(128) → BN → ReLU → Dropout
#       → Linear(6)
# ============================================================
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class MLPClassifier(nn.Module):
    def __init__(self, input_dim=480, num_classes=6, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.network(x)

def train_mlp(X_train, y_train, X_val, y_val, epochs=50, batch_size=64,
              lr=1e-3, device=device):
    model = MLPClassifier(input_dim=X_train.shape[1],
                          num_classes=len(np.unique(y_train))).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),
        batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val)),
        batch_size=batch_size)

    best_f1, history = 0.0, {"train_loss": [], "val_f1": []}
    os.makedirs("models", exist_ok=True)
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        model.eval()
        preds, labels_all = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                labels_all.extend(yb.cpu().numpy())
                preds.extend(model(xb).argmax(dim=1).cpu().numpy())
        val_f1 = f1_score(labels_all, preds, average="macro")
        history["train_loss"].append(total_loss / len(train_loader))
        history["val_f1"].append(val_f1)
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "models/best_mlp.pt")
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | Train Loss: "
                  f"{total_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f}")
    print(f"\nBest Val Macro F1: {best_f1:.4f}")
    return model, history

mlp_model, mlp_history = train_mlp(X_train, y_train, X_val, y_val)

In [ ]:
# ============================================================
# CELL 8: Evaluate MLP on held-out test set
# ============================================================
from torch.utils.data import DataLoader, TensorDataset

mlp_model.load_state_dict(torch.load("models/best_mlp.pt", map_location=device))
mlp_model.eval()

test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)),
    batch_size=64)
preds, labels_all = [], []
with torch.no_grad():
    for xb, _ in test_loader:
        preds.extend(mlp_model(xb.to(device)).argmax(dim=1).cpu().numpy())
labels_all = y_test

mlp_report = {
    "accuracy": accuracy_score(labels_all, preds),
    "f1_macro": f1_score(labels_all, preds, average="macro"),
    "report": classification_report(labels_all, preds, output_dict=True),
    "predictions": np.array(preds),
}

print("=== MLP TEST RESULTS ===")
print(f"Accuracy: {mlp_report['accuracy']:.4f}  Macro F1: {mlp_report['f1_macro']:.4f}")
print(classification_report(labels_all, preds, target_names=EC_CLASSES))

## Step 5: Results & Figures

In [ ]:
# ============================================================
# CELL 9: Model comparison bar chart
# ============================================================
import json

model_names = list(sklearn_results.keys()) + ["ESM-2 + MLP"]
f1_scores = [r["f1_macro"] for r in sklearn_results.values()] + [mlp_report["f1_macro"]]
accs = [r["accuracy"] for r in sklearn_results.values()] + [mlp_report["accuracy"]]

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(model_names)))
x = np.arange(len(model_names))
ax.bar(x, f1_scores, color=colors, width=0.5, label="Macro F1")
ax.bar(x, accs, color=colors, width=0.25, alpha=0.5, label="Accuracy")
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha="right")
ax.set_title("Model Comparison — Enzyme EC Classification (ESM-2 Embeddings)")
ax.set_ylabel("Score")
ax.legend()
for i, (f1, acc) in enumerate(zip(f1_scores, accs)):
    ax.text(i, f1 + 0.01, f"{f1:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CELL 10: Confusion matrix
# ============================================================
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels_all, preds)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=EC_CLASSES, yticklabels=EC_CLASSES, ax=ax)
ax.set_title("Confusion Matrix — ESM-2 + MLP")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.savefig("results/figures/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CELL 11: Training curves
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, len(mlp_history["train_loss"]) + 1)

axes[0].plot(epochs, mlp_history["train_loss"], color="#2196F3", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, mlp_history["val_f1"], color="#4CAF50", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Validation F1")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/figures/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CELL 12: Save all results to JSON
# ============================================================
results = {
    "sklearn_baselines": {},
    "pytorch_models": {"ESM-2 + MLP": {
        "accuracy": mlp_report["accuracy"],
        "f1_macro": mlp_report["f1_macro"],
    }},
    "training_history_mlp": mlp_history,
}
for name, r in sklearn_results.items():
    results["sklearn_baselines"][name] = {
        "accuracy": float(r["accuracy"]), "f1_macro": float(r["f1_macro"])
    }

with open("results/experiment_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n=== FINAL SUMMARY ===")
print(f"{'Model':25s}  {'Accuracy':>10s}  {'Macro F1':>10s}")
for name in model_names:
    if name in sklearn_results:
        r = sklearn_results[name]
    else:
        r = mlp_report
    print(f"{name:25s}  {r['accuracy']:10.4f}  {r['f1_macro']:10.4f}")

print("\nSaved: results/experiment_results.json")

## Done!

Download from Colab:
```
!zip -r results.zip results models enzymes_swissprot.csv esm2_embeddings.npy labels.npy
# then download results.zip from the Files panel
```

These results back your cold email to Prof. Xin Gao.